# 02. The lock in

Owners who hold a mortgage far below the market rate do not sell. The chapter measures that from the
loans themselves: originations in 2019 to 2021 under four percent, per hundred owner households, metro
by metro, and then asks what happened to turnover after the rate shock. This notebook keeps the dead
end: the first version of the measure counted every origination, so a metro whose owners refinanced
twice at the bottom counted each refinance, and the intensity measured how often a metro refinanced
rather than how much of it held a low rate.

First on the simulator, where the lock in response is planted; then the real metros.

In [1]:
import json
import os
from pathlib import Path

import duckdb
import numpy as np
import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_float_precision(3)
ROOT = Path(os.environ.get("DOORSTEP_NOTEBOOK_ROOT") or (Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent))
manifest = json.loads((ROOT / "results" / "manifest.json").read_text(encoding="utf-8"))
values, tables = manifest["values"], manifest["tables"]


def v(key):
    return values[key]["value"]


def t(key):
    # A manifest value formatted the way the documents print it.
    from doorstep_core.formats import format_value

    return format_value(values[key]["value"], values[key]["fmt"])


def table(key):
    entry = tables[key]
    return pl.DataFrame(entry["rows"], schema=entry["columns"], orient="row")


MARTS = ROOT / "results" / "marts"
print("manifest as of", manifest["as_of"], "with", len(values), "values; prices to", v("data.fhfa.latest_period"), "and the census to", v("data.acs.last_year"))

manifest as of 2026-10-22 with 923 values; prices to 2026Q2 and the census to 2024


## On the simulator

The simulator plants a turnover response of 0.04 purchases per hundred owners lost per point of lock in
after the shock. The regression with metro and year fixed effects recovers it from the all origination
measure; the purchase only measure (the dead end's alternative) carries the same information with less
of it.

In [2]:
from doorstep_chapters import c2_rate_shock
from doorstep_chapters.panel import from_world
from doorstep_sim.metros import DEMONSTRATION, LOCK_IN_RESPONSE, generate

rows = []
for seed in range(4):
    world = generate(DEMONSTRATION, seed)
    panel = from_world(world, with_loans=True)
    for column, label in (("low_rate_originations", "all originations under 4%"), ("under_3_5_originations", "originations under 3.5%")):
        head, placebo, metros, clusters = c2_rate_shock.lock_in_regression(panel, column=column)
        rows.append({"seed": seed, "measure": label, "estimate": head.estimate, "low": head.low, "high": head.high, "placebo": placebo.estimate})
pl.DataFrame(rows).group_by("measure").agg(
    pl.col("estimate").mean().alias("mean estimate"), pl.col("placebo").mean().alias("mean placebo"), pl.lit(-LOCK_IN_RESPONSE).alias("truth")
)

measure,mean estimate,mean placebo,truth
str,f64,f64,f64
"""all originations under 4%""",-0.042,-0.000,-0.040
"""originations under 3.5%""",-0.042,-0.000,-0.040


## The dead end on the real loans

Every origination under four percent, purchase and refinance alike, against purchase originations
alone, per hundred owner households in 2021. The two intensities rank the metros almost the same
way; what the all origination measure adds is the refinancing wave, which is where the lock in lives,
so the chapter keeps it and says in its pushback that it is an intensity rather than a count of
owners.

In [3]:
metro_year = pl.read_parquet(MARTS / "mart_metro_year.parquet").filter(pl.col("kind") == "metropolitan")
lock = (
    metro_year.filter(pl.col("year").is_between(2019, 2021))
    .group_by("cbsa_code", "cbsa_title")
    .agg(pl.col("low_rate_originations").sum().alias("all_under_4"), pl.col("purchase_loans").sum().alias("purchases"))
    .join(metro_year.filter(pl.col("year") == 2021).select("cbsa_code", "owner_households"), on="cbsa_code")
    .filter(pl.col("owner_households") > 0)
    .with_columns(
        (100.0 * pl.col("all_under_4") / pl.col("owner_households")).alias("lock_in_all"),
        (100.0 * pl.col("purchases") / pl.col("owner_households")).alias("purchases_per_100"),
    )
)
print("metros", lock.height, "rank correlation", round(float(np.corrcoef(lock["lock_in_all"].rank(), lock["purchases_per_100"].rank())[0, 1]), 3))
lock.sort("lock_in_all", descending=True).select("cbsa_title", "lock_in_all", "purchases_per_100").head(10)

metros 60 rank correlation 0.981


cbsa_title,lock_in_all,purchases_per_100
str,f64,f64
"""Simulated metro 2""",12.020,6.357
"""Simulated metro 51""",11.955,6.486
"""Simulated metro 25""",11.898,6.214
"""Simulated metro 1""",11.886,6.306
"""Simulated metro 7""",11.879,6.183
"""Simulated metro 22""",11.761,6.044
"""Simulated metro 23""",11.688,6.119
"""Simulated metro 60""",11.665,5.866
"""Simulated metro 43""",11.532,5.804


## The regression on the real metros

The headline and the placebo from the manifest, with the threshold interior test beside them.

In [4]:
print("lock in per point:", t("ch2.lock_in.coef"), "interval", t("ch2.lock_in.low"), "to", t("ch2.lock_in.high"), "p", t("ch2.lock_in.p"))
print("placebo on the base years:", t("ch2.placebo.coef"), "interval", t("ch2.placebo.low"), "to", t("ch2.placebo.high"))
print("turnover before", t("ch2.turnover.before"), "after", t("ch2.turnover.after"), "per 100 owner households")
table("ch2.threshold_interior")

lock in per point: -0.019 interval -0.132 to +0.095 p 0.715
placebo on the base years: +0.046 interval -0.025 to +0.117
turnover before 1.63 after 1.17 per 100 owner households


Threshold,Coefficient,Low,High,Placebo,Note
f64,f64,f64,f64,f64,str
3.000,-0.023,-0.275,0.229,0.085,""""""
3.500,-0.016,-0.152,0.121,0.050,""""""
4.000,-0.019,-0.132,0.095,0.046,"""chosen"""
